# CRISP-DM Case 1: 2330.TW Stock Price Prediction

這是一個使用 CRISP-DM 方法論解決時間序列迴歸 (Time Series Regression) 問題的教學範例。我們將使用台積電 (2330.TW) 的歷史股價資料，建立模型來預測未來的收盤價。

## Step 1: Load Data + Generate Random Data Scatter Plot

這一步是先把資料讀進來，並用圖表觀察資料長什麼樣子。
在機器學習中，不要一開始就急著訓練模型，先看懂資料，比直接建模更重要。

### 產生 Random Regression Data Scatter Plot
這是一個簡單的隨機資料散佈圖，讓我們先理解「迴歸」的概念：找到一條線或曲線，讓模型可以根據 X 預測 Y。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 隨機產生 X 和 Y
x = np.random.uniform(-100, 100, 200)
a = np.random.uniform(-50, 50)
b = np.random.uniform(0, 100)
noise = np.random.normal(0, 300, 200)

y = a * x + b + noise

plt.scatter(x, y)
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Random Regression Data Scatter Plot")
plt.show()

### 載入台積電股價資料

In [ ]:
import yfinance as yf
import pandas as pd
import os

# 下載資料
ticker = "2330.TW"
df = yf.download(ticker, start="2020-01-01", end="2026-01-01")
df.head()

In [ ]:
# 畫出收盤價走勢圖與成交量
plt.figure(figsize=(14, 6))

plt.subplot(2, 1, 1)
plt.plot(df.index, df['Close'])
plt.title(f"{ticker} Close Price")
plt.ylabel("Price")

plt.subplot(2, 1, 2)
plt.bar(df.index, df['Volume'])
plt.title(f"{ticker} Volume")
plt.ylabel("Volume")

plt.tight_layout()
plt.show()

## Step 2: Preprocessing

資料前處理是把原始資料整理成模型看得懂的格式。
例如文字類別要轉成數字，缺失值要處理，資料型態要正確，訓練集與測試集也要分開。

In [ ]:
# 處理缺失值
df = df.dropna()

# 特徵工程 (Feature Engineering)
df["MA_5"] = df["Close"].rolling(window=5).mean()
df["MA_10"] = df["Close"].rolling(window=10).mean()
df["MA_20"] = df["Close"].rolling(window=20).mean()

df["Return"] = df["Close"].pct_change()

# 延遲特徵 (Lag features) - 使用過去的價格來預測未來
df["Lag_1_Close"] = df["Close"].shift(1)
df["Lag_2_Close"] = df["Close"].shift(2)
df["Lag_3_Close"] = df["Close"].shift(3)

# 移除因為 rolling 和 shift 產生的 NaN
df = df.dropna()

# 準備 X 和 y
features = ["Open", "High", "Low", "Volume", "MA_5", "MA_10", "MA_20", "Return", "Lag_1_Close", "Lag_2_Close", "Lag_3_Close"]
X = df[features]
y = df["Close"]

# 轉換型態
X = X.astype(float)
y = y.astype(float)

# 儲存處理好的資料供後續使用
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/stock_2330_processed.csv")

### Train Test Split (時間序列切分)

Train Set 用來訓練模型。Test Set 用來檢查模型是否真的能預測沒看過的資料。
如果只用同一份資料訓練與測試，模型可能只是記住答案，而不是真的學到規律。

**注意**: 股價資料不可隨機切分，需保留時間順序。

In [ ]:
# 時間序列的 Train/Test Split
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
y_train = y.iloc[:split_index]
X_test = X.iloc[split_index:]
y_test = y.iloc[split_index:]

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

## Step 3: Build Model

建模就是讓模型從資料中學習 X 和 y 之間的關係。
在迴歸問題中，模型的目標是預測一個連續數值。

In [ ]:
from sklearn.linear_model import LinearRegression

# 建立並訓練模型
model = LinearRegression()
model.fit(X_train, y_train)

# 預測
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

## Step 4: Evaluation

模型訓練完後，不能只看預測結果感覺準不準，需要用 MSE、MAE、R² 等指標客觀評估。
也要觀察模型是否 overfit 或 underfit。

- **MSE**: 平均平方誤差。數值越小，代表模型預測越準。會放大大錯誤。
- **MAE**: 平均絕對誤差。可以理解成平均預測差多少錢。
- **R²**: 決定係數。越接近 1，代表模型解釋能力越好。

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 計算測試集指標
mse = mean_squared_error(y_test, y_pred_test)
mae = mean_absolute_error(y_test, y_pred_test)
r2 = r2_score(y_test, y_pred_test)

train_r2 = r2_score(y_train, y_pred_train)

print(f"Train R²: {train_r2:.4f}")
print(f"Test R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"MAE: {mae:.4f}")

### 觀察預測曲線與 Overfitting / Underfitting 判斷

- **Overfitting**: Train R² 高，Test R² 低。
- **Underfitting**: Train R² 低，Test R² 也低。
- **Good Fit**: Train R² 和 Test R² 都高且接近。

In [ ]:
# 畫出預測曲線與實際曲線的比較
plt.figure(figsize=(14, 6))

# 我們把 train 和 test 接在一起畫
plt.plot(df.index[:split_index], y_train, label="Actual Train")
plt.plot(df.index[split_index:], y_test, label="Actual Test")
plt.plot(df.index[split_index:], y_pred_test, label="Predicted Test", linestyle="--")

plt.title("Train / Test / Predicted Curve")
plt.xlabel("Date")
plt.ylabel("Close Price")
plt.legend()
plt.show()

### Optuna 模型優化 (以 Random Forest 為例)

In [ ]:
import optuna
from sklearn.ensemble import RandomForestRegressor

def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 150)
    max_depth = trial.suggest_int("max_depth", 2, 10)

    rf_model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict(X_test)
    
    return mean_squared_error(y_test, rf_pred)

# 為了加快示範，只跑 10 次 trials
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10)

print("Best params:", study.best_params)

# 使用最佳參數重新訓練模型
best_rf = RandomForestRegressor(**study.best_params, random_state=42)
best_rf.fit(X_train, y_train)

rf_pred_test = best_rf.predict(X_test)
print(f"Optimized Random Forest Test R²: {r2_score(y_test, rf_pred_test):.4f}")

## Step 5: Deployment

部署是把訓練好的模型保存起來，讓其他程式或使用者可以輸入新資料並取得預測結果。
Pickle 可以儲存模型，Gzip 可以壓縮模型檔案大小。

In [ ]:
import pickle
import gzip
import os

# 確保 models 目錄存在
os.makedirs("../models", exist_ok=True)

# 儲存 Linear Regression 模型 (使用 gzip 壓縮)
model_path = "../models/stock_model.pkl.gz"
with gzip.open(model_path, "wb") as f:
    pickle.dump(model, f)
    
print(f"Model saved to {model_path}")

# 測試載入模型
with gzip.open(model_path, "rb") as f:
    loaded_model = pickle.load(f)

# 驗證載入的模型是否可用
sample_input = X_test.iloc[[0]]
print("Sample prediction:", loaded_model.predict(sample_input))